In [1]:
%%bash
cat << 'EOF' > /content/config.sh
#!/bin/bash

export ROOTDIR="/content/motos"
export VIDEOSOURCE="gsplat/input/IMG_4806.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=5

#branche dev
export GIT_BRANCH="dev"
export BASENAME="motos"

export PREPROCESS_PROFILE="colmap"
export GSPLAT_PROFILE="quality"

EOF

In [2]:
!cat /content/config.sh

#!/bin/bash

export ROOTDIR="/content/motos"
export VIDEOSOURCE="gsplat/input/IMG_4806.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=5

#branche dev
export GIT_BRANCH="dev"
export BASENAME="motos"

export PREPROCESS_PROFILE="colmap"
export GSPLAT_PROFILE="quality"



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /usr/local/miniforge

# activer conda pour cette session notebook
import os
os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

!conda --version

PREFIX=/usr/local/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.4.22-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_18.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_102.conda
Extracting libgcc-15.2.0-he0feb66_18.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.7.5-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_18.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.3-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libstdcxx-15.2.0-h934c35e_1

In [5]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
(wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh && \
chmod +x Miniconda3-latest-Linux-x86_64.sh  && \
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda  && \
/usr/local/miniconda/bin/conda init bash)

skipping this stage


In [6]:
%%bash
set -e
source /content/config.sh

mkdir -p "$ROOTDIR"

if [ -n "$VIDEOSOURCE" ]; then
  SRC_VIDEO="/content/drive/MyDrive/$VIDEOSOURCE"

  if [ ! -f "$SRC_VIDEO" ]; then
    echo "❌ Video not found: $SRC_VIDEO"
  else
    echo "🎬 Copying video: $SRC_VIDEO"
    cp -f "$SRC_VIDEO" "$ROOTDIR/video.mp4"
  fi
fi

if [ -n "$IMAGESET" ]; then
  SRC_IMAGES="/content/drive/MyDrive/$IMAGESET"
  DST_IMAGES="$ROOTDIR/images"

  if [ ! -d "$SRC_IMAGES" ]; then
    echo "❌ Image dataset not found: $SRC_IMAGES"
  else
    echo "🖼️ Preparing image dataset: $SRC_IMAGES"

    mkdir -p "$DST_IMAGES"

    COUNT=$(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l | tr -d ' ')
    if [ "$COUNT" -lt 2 ]; then
      echo "❌ Not enough images ($COUNT)"
      exit 1
    fi

    rm -f "$DST_IMAGES"/frame_*.png 2>/dev/null || true

    i=1
    for img in $(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | sort); do
      printf -v idx "%05d" "$i"
      cp "$img" "$DST_IMAGES/frame_${idx}.png"
      i=$((i+1))
    done

    echo "✅ Dataset ready in $DST_IMAGES"
  fi
fi

🎬 Copying video: /content/drive/MyDrive/gsplat/input/IMG_4806.MOV
❌ Image dataset not found: /content/drive/MyDrive/gsplat/input/perfume/video


In [7]:
!git clone https://github.com/NicoIGN/video_to_ply.git
%cd video_to_ply

Cloning into 'video_to_ply'...
remote: Enumerating objects: 1337, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 1337 (delta 52), reused 83 (delta 31), pack-reused 1229 (from 2)
Receiving objects: 100% (1337/1337), 506.99 KiB | 2.56 MiB/s, done.
Resolving deltas: 100% (828/828), done.
/content/video_to_ply


In [8]:
%%bash
cd /content/video_to_ply
source /content/config.sh
git stash save && git checkout $GIT_BRANCH && git pull

No local changes to save
Your branch is up to date with 'origin/dev'.
Already up to date.


Already on 'dev'


In [9]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && mamba env remove -y -n gsplat )

skipping this stage


In [10]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
mamba env list | grep -q gsplat && \
cd /content/video_to_ply/ && \
mamba run -n gsplat mamba env update -n gsplat -f environment/conda_colab.yml --prune -y || \
mamba env create -n gsplat -f environment/conda_colab.yml -y

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
Extracting  (121)  ⣾  [+] 2m:46.6s
Extracting  (121)  ⣾  [+] 2m:46.7s
Extracting  (121)  ⣾  [+] 2m:46.8s
Extracting  (121)  ⣾  [+] 2m:46.9s
Extracting  (121)  ⣾  [+] 2m:47.0s
Extracting  (121)  ⣾  [+] 2m:47.1s
Extracting  (121)  ⣾  [+] 2m:47.2s
Extracting  (121)  ⣾  [+] 2m:47.3s
Extracting  (121)  ⣾  [+] 2m:47.4s
Extracting  (121)  ⣾  [+] 2m:47.5s
Extracting  (121)  ⣾  [+] 2m:47.6s
Extracting  (121)  ⣾  [+] 2m:47.7s
Extracting  (121)  ⣾  [+] 2m:47.8s
Extracting  (121)  ⣾  [+] 2m:47.9s
Extracting  (121)  ⣾  [+] 2m:48.0s
Extracting  (121)  ⣾  [+] 2m:48.1s
Extracting  (121)  ⣾  [+] 2m:48.2s
Extracting  (121)  ⣾  [+] 2m:48.3s
Extracting  (121)  ⣾  [+] 2m:48.4s
Extracting  (121)  ⣾  [+] 2m:48.5s
Extracting  (121)  ⣾  [+] 2m:48.6s
Extracting  (121)  ⣾  [+] 2m:48.7s
Extracting  (121)  ⣾  [+] 2m:48.8s
Extracting  (121)  ⣾  [+] 2m:48.9s
Extracting  (121)  ⣾  [+] 2m:49.0s
Extracting  (121)  ⣾  [+] 2m:49.1s
Extracting  

In [11]:
%%bash
source /usr/local/miniforge/etc/profile.d/conda.sh

SITE_PACKAGES=$(mamba run -n gsplat python -c "import site; print(site.getsitepackages()[0])")
TARGET="$SITE_PACKAGES/SuperGluePretrainedNetwork"

if [ ! -d "$TARGET" ]; then
    git clone --depth 1 \
        https://github.com/magicleap/SuperGluePretrainedNetwork.git \
        "$TARGET"
else
    echo "SuperGluePretrainedNetwork already installed."
fi

Cloning into '/usr/local/miniforge/envs/gsplat/lib/python3.10/site-packages/SuperGluePretrainedNetwork'...


In [ ]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh && \
cd /content/video_to_ply/ && \
mamba run -n gsplat bash run.sh  \
--root "$ROOTDIR" \
--name "$BASENAME" \
--video $ROOTDIR/video.mp4  \
--preprocess-profile "$PREPROCESS_PROFILE" \
--gsplat-profile "$GSPLAT_PROFILE" \
--fps $FPS \
--skip-conda \
--skip-filter \
--no-proxy

🚫 Proxy disabled (NO_PROXY=true)
👉 using profile: preprocess/colmap
👉 using profile: gsplat/quality
⏩ Skipping conda setup (--skip-conda enabled)
✅ Using python: Python 3.10.20 
🚀 GPU model OK: splatfacto
📦 ROOT: /content/motos
🎬 Extracting frames at 5 FPS → /content/motos/ori/images
🎬 Video:   /content/motos/input/video.mov
📏 Width:   1280px
📁 Output:  /content/motos/ori/images
⚙️ Mode:    Fixed FPS
🎞️ FPS:     5
frame=  306 fps=4.8 q=-0.0 Lsize=N/A time=00:01:01.20 bitrate=N/A speed=0.961x    
✅ Extracted 306 frames


🧭 Running PREPROCESS through NerfStudio...
📝 Process log: /content/motos/ori/logs/ns_process.log
💓 Heartbeat log: /content/motos/ori/logs/ns_process_heartbeat.log
────────────────────────────────────
📁 INPUT                 : /content/motos/ori/images
📁 OUTPUT                : /content/motos/ori
⚙️ DEVICE                : gpu
📷 CAMERA TYPE           : perspective
🔀 MATCHING METHOD       : exhaustive
🧠 SFM TOOL             : colmap
🧬 FEATURE TYPE          : any
🔗 MATCHER

In [ ]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && mamba run -n gsplat tree /content/exterieur/model3d


In [14]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && \
  source /content/config.sh && \
  cd /content/video_to_ply/ && \
  INPUT_ARG="" && \
  if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
    INPUT_ARG="--images $ROOTDIR/images --name $BASENAME"; \
  elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
    INPUT_ARG="--video $ROOTDIR/video.mp4 --name $BASENAME"; \
  fi && \
  mamba run -n gsplat bash run.sh $INPUT_ARG \
    --root "$ROOTDIR" \
    --skip-conda \
    --profile "$PROFILE" \
    --no-proxy \
    --skip-conda \
    --skip-frame-extraction \
    --skip-colmap \
    --skip-training )

skipping this stage


In [ ]:
from google.colab import files
import os
import subprocess

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)


# =========================
# EXPORT ORIGINAL PLY
# =========================
base_ply = os.path.join(export_dir, f"{basename}.ply")
if os.path.exists(base_ply):
    print(f"⬇️ Downloading original PLY: {os.path.basename(base_ply)}")
    files.download(base_ply)
else:
    print(f"⚠️ Original PLY not found: {base_ply}")


# =========================
# EXPORT CONFIG
# =========================
training_config = os.path.join(export_dir, f"config.yml")
if os.path.exists(training_config):
    print(f"⬇️ Downloading training config: {os.path.basename(training_config)}")
    files.download(training_config)
else:
    print(f"⚠️ Config not found: {training_config}")


In [ ]:
from google.colab import files
import os
import subprocess
import glob

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)

# =========================
# FIND FILTERED PLYS
# =========================
filtered_plys = sorted(
    glob.glob(os.path.join(export_dir, f"{basename}_*.ply"))
)

# =========================
# EXPORT FILTERED PLYS
# =========================
if not filtered_plys:
    print("⚠️ No filtered PLY files found.")
    print(f"📂 Searched: {export_dir}")
else:
    print(f"📦 Found {len(filtered_plys)} filtered PLY file(s)")

    for ply_path in filtered_plys:
        print(f"⬇️ Downloading filtered PLY: {os.path.basename(ply_path)}")
        files.download(ply_path)